# Phase 

$$ 
 \frac{r_{A1} + r_{A2}}{2} = r_B , \quad r_{A1} > r_B , \quad r_{A2} < r_B
$$ 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'


@njit
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((alpha * (n_2 - n_1) + n_1))))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

def find_last_maximum(array, t_array, percentile):
    indices = np.where(array >= percentile)[0]
    if len(indices) > 0:
        return (t_array[indices[-1]], array[indices[-1]])
    return None

@njit
def integrate_sc1(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T):
    num_steps = int(tf / dt)
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        t_prev = (i - 1) * dt
        alpha = alpha_0 * (np.cos((np.pi * t_prev) / T))**2
        beta  = alpha_0 * (np.cos((np.pi * t_prev) / T))**2
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta) * dt
    return n_1, n_2

def calculate_phase_sc1(alpha_0, r_A1, r_A2, T):
    n01, n02 = 0.01, 0.0
    tf, dt = 99600.0, 0.1
    r_B = d_A = d_B = 1.0
    n_1, n_2 = integrate_sc1(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T)
    t = np.linspace(0.0, tf, len(n_1))
    shifted_n_2 = n_2 + (n_1.max() - n_2.max())
    time_range = (t >= 60000) & (t <= tf)
    if len(shifted_n_2[time_range]) == 0:
        return np.nan
    perc_n1 = np.percentile(n_1, 99)
    perc_n2 = np.percentile(shifted_n_2[time_range], 99)
    last1 = find_last_maximum(n_1, t, perc_n1)
    last2 = find_last_maximum(shifted_n_2, t, perc_n2)
    if last1 is None or last2 is None:
        return np.nan
    dt_diff = np.abs(last1[0] - last2[0])
    dt_diff = min(dt_diff % T, T - (dt_diff % T))
    return 2 * np.pi * (dt_diff / T)

@njit
def integrate_sc2(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T):
    num_steps = int(tf / dt)
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        t_prev = (i - 1) * dt
        alpha = alpha_0 * (np.cos((np.pi * t_prev) / T))**2
        beta  = alpha_0 * (np.sin((np.pi * t_prev) / T))**2
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta) * dt
    return n_1, n_2

def calculate_phase_sc2(alpha_0, r_A1, r_A2, T):
    n01, n02 = 0.01, 0.0
    tf, dt = 99700.0, 0.1
    r_B = d_A = d_B = 1.0
    n_1, n_2 = integrate_sc2(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T)
    t = np.linspace(0.0, tf, len(n_1))
    shifted_n_2 = n_2 + (n_1.max() - n_2.max())
    time_range = (t >= 60000) & (t <= tf)
    if len(shifted_n_2[time_range]) == 0:
        return np.nan
    perc_n1 = np.percentile(n_1, 85)
    perc_n2 = np.percentile(shifted_n_2[time_range], 95)
    last1 = find_last_maximum(n_1, t, perc_n1)
    last2 = find_last_maximum(shifted_n_2, t, perc_n2)
    if last1 is None or last2 is None:
        return np.nan
    dt_diff = np.abs(last1[0] - last2[0])
    dt_diff = min(dt_diff % T, T - (dt_diff % T))
    return 2 * np.pi * (dt_diff / T)

@njit
def integrate_sc3(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T, beta_0_fixed=0.1):
    num_steps = int(tf / dt)
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        t_prev = (i - 1) * dt
        alpha = alpha_0 * (np.sin((np.pi * t_prev) / T))**2
        beta  = beta_0_fixed * (np.sin((np.pi * t_prev) / T))**2
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta) * dt
    return n_1, n_2

def calculate_phase_sc3(alpha_0, r_A1, r_A2, T):
    n01, n02 = 0.01, 0.0
    tf, dt = 100100.0, 0.1
    r_B = d_A = d_B = 1.0
    n_1, n_2 = integrate_sc3(n01, n02, tf, dt, r_A1, r_A2, r_B, d_A, d_B, alpha_0, T)
    t = np.linspace(0.0, tf, len(n_1))
    shifted_n_2 = n_2 + (n_1.max() - n_2.max())
    time_range = (t >= 60000) & (t <= tf)
    if len(shifted_n_2[time_range]) == 0:
        return np.nan
    perc_n1 = np.percentile(n_1, 99)
    perc_n2 = np.percentile(shifted_n_2[time_range], 99)
    last1 = find_last_maximum(n_1, t, perc_n1)
    last2 = find_last_maximum(shifted_n_2, t, perc_n2)
    if last1 is None or last2 is None:
        return np.nan
    dt_diff = np.abs(last1[0] - last2[0])
    dt_diff = min(dt_diff % T, T - (dt_diff % T))
    return 2 * np.pi * (dt_diff / T)

T_values = [50, 500, 5000]
r_A1_values = [1.1, 1.2, 1.3, 1.4]
r_A2_values = [0.9, 0.8, 0.7, 0.6]

print("=" * 60)
print("Scenario 1: \\(\\alpha=\\cos^2, \\beta=\\cos^2\\)")
print("=" * 60)
alpha_0_sc1 = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
_ = calculate_phase_sc1(0.01, 1.1, 0.9, 50)
all_heatmaps_sc1 = []
for T in T_values:
    print(f"  T = {T}")
    delta_r = sorted(set([round(a - b, 1) for a, b in zip(r_A1_values, r_A2_values)]))
    data = np.zeros((len(delta_r), len(alpha_0_sc1)))
    for i, dr in enumerate(delta_r):
        for j, mu0 in enumerate(alpha_0_sc1):
            for r1, r2 in zip(r_A1_values, r_A2_values):
                if round(r1 - r2, 1) == dr:
                    data[i, j] = calculate_phase_sc1(mu0, r1, r2, T)
                    break
    all_heatmaps_sc1.append({'data': data, 'delta_r': delta_r, 'T': T})
    np.save(f'heatmap_sc1_T{T}.npy', data)

print("=" * 60)
print("Scenario 2: \\(\\alpha=\\cos^2, \\beta=\\sin^2\\)")
print("=" * 60)
alpha_0_sc2 = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
_ = calculate_phase_sc2(0.05, 1.1, 0.9, 50)
all_heatmaps_sc2 = []
for T in T_values:
    print(f"  T = {T}")
    delta_r = sorted(set([round(a - b, 1) for a, b in zip(r_A1_values, r_A2_values)]))
    data = np.zeros((len(delta_r), len(alpha_0_sc2)))
    for i, dr in enumerate(delta_r):
        for j, mu0 in enumerate(alpha_0_sc2):
            for r1, r2 in zip(r_A1_values, r_A2_values):
                if round(r1 - r2, 1) == dr:
                    data[i, j] = calculate_phase_sc2(mu0, r1, r2, T)
                    break
    all_heatmaps_sc2.append({'data': data, 'delta_r': delta_r, 'T': T})
    np.save(f'heatmap_sc2_T{T}.npy', data)

print("=" * 60)
print("Scenario 3: \\(\\alpha=\\sin^2\\) (var), \\(\\beta=0.1\\times\\sin^2\\)")
print("=" * 60)
alpha_0_sc3 = [0.15, 0.2, 0.3]
_ = calculate_phase_sc3(0.15, 1.1, 0.9, 50)
all_heatmaps_sc3 = []
for T in T_values:
    print(f"  T = {T}")
    delta_r = sorted(set([round(a - b, 1) for a, b in zip(r_A1_values, r_A2_values)]))
    data = np.zeros((len(delta_r), len(alpha_0_sc3)))
    for i, dr in enumerate(delta_r):
        for j, mu0 in enumerate(alpha_0_sc3):
            for r1, r2 in zip(r_A1_values, r_A2_values):
                if round(r1 - r2, 1) == dr:
                    data[i, j] = calculate_phase_sc3(mu0, r1, r2, T)
                    break
    all_heatmaps_sc3.append({'data': data, 'delta_r': delta_r, 'T': T})
    np.save(f'heatmap_sc3_T{T}.npy', data)

print("\nAll calculations finished. Creating combined figure...")

fig, axes = plt.subplots(3, 3, figsize=(17, 13))

fig.subplots_adjust(wspace=0.1, hspace=0.17,
                    left=0.10, right=0.88, top=0.90, bottom=0.08)

scenarios = [
    (all_heatmaps_sc1, alpha_0_sc1),
    (all_heatmaps_sc2, alpha_0_sc2),
    (all_heatmaps_sc3, alpha_0_sc3),
]

all_data_global = np.concatenate(
    [h['data'].ravel() for hm in [all_heatmaps_sc1, all_heatmaps_sc2, all_heatmaps_sc3] for h in hm]
)
vmin = 0.0
vmax = np.pi

ims = []
for row, (heatmaps, mu_vals) in enumerate(scenarios):
    for col, h in enumerate(heatmaps):
        ax = axes[row, col]
        
        ax.grid(False)
        
        im = ax.imshow(h['data'], cmap='viridis', aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax)
        ims.append(im)

       
        ax.set_xticks(range(len(mu_vals)))
        ax.set_xticklabels([f'{x:.2f}' for x in mu_vals],
                           fontsize=15, rotation=45, ha='right')

        
        ax.set_yticks(range(len(h['delta_r'])))
        ax.set_yticklabels([f'{x:.1f}' for x in h['delta_r']], 
                           fontsize=15)

       
        if row == 0:
            ax.set_title(rf'\( T = {h["T"]} \)', fontsize=20, pad=6)


for i, lab in enumerate(['A , I', 'B , I', 'C , I']):
    fig.text(0.02, 0.78 - i*0.30, lab, fontsize=27, ha='center', va='center',
             fontfamily='serif', style='italic')


row_labels = [r'\( \delta r \)', r'\( \delta r \)', r'\( \delta r \)']
for row in range(3):
    y_pos = 0.76 - row * 0.28  
    fig.text(0.07, y_pos, row_labels[row], fontsize=25,
             rotation=90, va='center', ha='center', fontfamily='serif')

fig.text(0.50, 0.015, r'\( \alpha_0 \)', fontsize=25,
         ha='center', va='center', fontfamily='serif')


cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
cbar = fig.colorbar(ims[0], cax=cax)
cbar.set_label(r'\( \delta\theta \)', fontsize=20, labelpad=8, fontfamily='serif')
cbar.ax.tick_params(labelsize=15)


from matplotlib.ticker import FuncFormatter
def pi_format(x, pos):
    if x == 0:
        return '0'
    elif x == np.pi/4:
        return r'\(\pi/4\)'
    elif x == np.pi/2:
        return r'\(\pi/2\)'
    elif x == 3*np.pi/4:
        return r'\(3\pi/4\)'
    elif x == np.pi:
        return r'\(\pi\)'
    else:
        return f'{x/np.pi:.2f}\(\pi\)'

cbar.ax.yaxis.set_major_formatter(FuncFormatter(pi_format))
cbar.ax.yaxis.set_major_locator(plt.MultipleLocator(np.pi/4))


plt.savefig('heatmap_all_scenarios.png', dpi=160, bbox_inches='tight', pad_inches=0.15)
plt.savefig('heatmap_all_scenarios.pdf', bbox_inches='tight', pad_inches=0.15)

plt.show()

# Time_averaged mutant frequency ($\bar{\phi}$) 

$$ 
 \frac{r_{A1} + r_{A2}}{2} = r_B , \quad r_{A1} > r_B , \quad r_{A2} < r_B
$$ 

# Symmetric In_phase 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2 - r  
oscillation_alpha = 'cos2'
oscillation_beta = 'cos2'
has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    
    for alpha0 in alpha0_values:
        limiting = None
        
        
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, alpha0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        
        avgs = []
        for T in T_values:
            
            if oscillation_alpha == 'cos2':
                alpha = alpha0 * (np.cos(np.pi * t / T))**2
            else:
                alpha = alpha0 * (np.sin(np.pi * t / T))**2
            
            if oscillation_beta == 'cos2':
                beta = alpha0 * (np.cos(np.pi * t / T))**2
            else:
                beta = alpha0 * (np.sin(np.pi * t / T))**2

           
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

       
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

        
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    
    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    
    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)


fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')


axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_in_phase_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Symmetric_out_of_phase

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2 - r


oscillation_alpha = 'cos2'
oscillation_beta = 'sin2'
has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, alpha0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

       
        avgs = []
        for T in T_values:
            
            if oscillation_alpha == 'cos2':
                alpha = alpha0 * (np.cos(np.pi * t / T))**2
            else:
                alpha = alpha0 * (np.sin(np.pi * t / T))**2
            
            if oscillation_beta == 'sin2':
                beta = alpha0 * (np.sin(np.pi * t / T))**2
            else:
                beta = alpha0 * (np.cos(np.pi * t / T))**2

        
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
           
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

       
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)


fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_out_of_phase_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

$$ 
\frac{r_{A1} + r_{A2}}{2} < r_{B} ,\qquad  r_{A1} > r_B ,\qquad r_{A2} < r_B 
$$  

# Symmetric In-phase

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r

has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
      
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, alpha0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        avgs = []
        for T in T_values:
            
            alpha = alpha0 * (np.cos((np.pi * t) / T))**2
            beta = alpha0 * (np.cos((np.pi * t) / T))**2

           
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

        
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)


fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_in_phase_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Symmetric Out-of-phase 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01 
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)

r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r  

oscillation_alpha = 'cos2'
oscillation_beta = 'sin2'
has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, alpha0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        avgs = []
        for T in T_values:
            
            if oscillation_alpha == 'cos2':
                alpha = alpha0 * (np.cos(np.pi * t / T))**2
            else:
                alpha = alpha0 * (np.sin(np.pi * t / T))**2
            
            if oscillation_beta == 'sin2':
                beta = alpha0 * (np.sin(np.pi * t / T))**2
            else:
                beta = alpha0 * (np.cos(np.pi * t / T))**2

           
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')


        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

    
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_out_of_phase_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

$$ 
\frac{r_{A1} + r_{A2}}{2} > r_{B} ,\qquad  r_{A1} > r_B ,\qquad r_{A2} <= r_B 
$$  

# Symmetric In-phase 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)

r_A1_values = [1.6, 1.7, 1.8]
r_A2_func = lambda r: 2.5 - r

has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []

for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, alpha0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

       
        avgs = []
        for T in T_values:

            alpha = alpha0 * (np.cos((np.pi * t) / T))**2
            beta = alpha0 * (np.cos((np.pi * t) / T))**2

           
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

       
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

       
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

    
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)


fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_in_phase_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Symmetric Out-of-phase 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.6, 1.7, 1.8]
r_A2_func = lambda r: 2.5 - r

has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
        avgs = []
        for T in T_values:
            
            alpha = alpha0 * (np.cos((np.pi * t) / T))**2
            beta = alpha0 * (np.sin((np.pi * t) / T))**2

            
            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

      
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

  
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)


fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('symmetric_out_of_phase_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')

plt.show()

$$ 
 \frac{r_{A1} + r_{A2}}{2} = r_B , \quad r_{A1} > r_B , \quad r_{A2} < r_B
$$ 

# Asymmetric In-phase -- More Migration towards deleterious island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01 
beta_0 = 0.1
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2.0 - r


has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
       
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, beta_0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

    
        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

        
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_alpha_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric In-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01 
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2.0 - r


has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None
        
      
        if has_constant_migration:
            ac = np.full(num_steps, alpha_0)
            bc = np.full(num_steps, beta0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        
        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

       
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

      
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[beta0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[beta0], limiting, r'$\infty$',
                    color=colors[beta0], fontsize=22, ha='center', va='center', alpha=0.85)

 
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_beta_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards deleterious island  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2

T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01  
beta_0 = 0.1
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2.0 - r


has_constant_migration = False 

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None

        
        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

  
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_alpha_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01 
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 2.0 - r


has_constant_migration = False  

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None


        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

    
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

  
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_beta_1.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

$$ 
\frac{r_{A1} + r_{A2}}{2} < r_{B} ,\qquad  r_{A1} > r_B ,\qquad r_{A2} < r_B 
$$  

# Asymmetric In-phase -- More Migration towards deleterious island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2

T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
beta_0 = 0.1 
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r 


has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
       
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, beta_0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

       
        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

    
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

        
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

    
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_alpha_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric In-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)

r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r  


has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None
        

        if has_constant_migration:
            ac = np.full(num_steps, alpha_0)
            bc = np.full(num_steps, beta0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

    
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

       
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[beta0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[beta0], limiting, r'$\infty$',
                    color=colors[beta0], fontsize=22, ha='center', va='center', alpha=0.85)

    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_beta_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards deleterious island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
beta_0 = 0.1
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r  


has_constant_migration = False 

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
      
        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_alpha_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.1, 1.2, 1.3]
r_A2_func = lambda r: 1.5 - r  

has_constant_migration = False  

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None
  
        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_beta_2.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

$$ 
\frac{r_{A1} + r_{A2}}{2} > r_{B} ,\qquad  r_{A1} > r_B ,\qquad r_{A2} < r_B 
$$  

# Asymmetric In-phase -- More Migration towards deleterious island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2

T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01 
beta_0 = 0.1
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.6, 1.7, 1.8]
r_A2_func = lambda r: 2.5 - r 

has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None
        
        
        if has_constant_migration:
            ac = np.full(num_steps, alpha0)
            bc = np.full(num_steps, beta_0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

       
        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

        
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[alpha0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[alpha0], limiting, r'$\infty$',
                    color=colors[alpha0], fontsize=22, ha='center', va='center', alpha=0.85)

   
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_alpha_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric In-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2

T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.6, 1.7, 1.8]  
r_A2_func = lambda r: 2.5 - r  


has_constant_migration = True

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None
        
    
        if has_constant_migration:
            ac = np.full(num_steps, alpha_0)
            bc = np.full(num_steps, beta0)
            n1c, n2c = integrate_euler(n01, n02, ac, bc, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            yc = (n1c + n2c) / 2
            mask = (t >= 60000) & (t <= tf)
            yr = yc[mask]
            if len(yr) > 0 and np.all(np.isfinite(yr)):
                limiting = 0.5 * (np.min(yr) + np.max(yr))
                if np.isfinite(limiting):
                    all_y.append(limiting)

        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.cos(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

       
        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

       
        if limiting is not None and np.isfinite(limiting):
            ax.axhline(limiting, color=colors[beta0], lw=2.2, alpha=0.85)
            ax.text(max(T_values)*inf_x_pos[beta0], limiting, r'$\infty$',
                    color=colors[beta0], fontsize=22, ha='center', va='center', alpha=0.85)

    
    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    if has_constant_migration:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*max(inf_x_pos.values())*1.4)
    else:
        ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Inphase_beta_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards deleterious island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
beta_0 = 0.1
alpha0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)


r_A1_values = [1.6, 1.7, 1.8]  
r_A2_func = lambda r: 2.5 - r  

has_constant_migration = False

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []


for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for alpha0 in alpha0_values:
        limiting = None

        avgs = []
        for T in T_values:
            alpha = alpha0 * (np.cos(np.pi * t / T))**2
            beta = beta_0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[alpha0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[alpha0], color=colors[alpha0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[alpha0], marker=markers[alpha0],
                           lw=2.5, markersize=12, markerfacecolor=colors[alpha0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \alpha_0 = {alpha0} \)')

    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_alpha_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Asymmetric Out-of-phase -- More Migration towards advantageous island 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from numba import njit
from matplotlib import rcParams

plt.style.use('seaborn-v0_8-whitegrid')
rcParams['font.family'] = 'serif'
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 18
rcParams['legend.fontsize'] = 18
rcParams['figure.dpi'] = 150
rcParams['savefig.dpi'] = 300
rcParams['text.usetex'] = True
rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

@njit(cache=True)
def f_1(n_1, n_2, r_A1, r_A2, r_B, d_A, d_B, alpha):
    return (((d_B * (1 - n_1) * ((1 - alpha) * r_A1 * n_1 + alpha * r_A2 * n_2)) -
             (r_B * d_A * n_1 * (1 - ((1 - alpha) * n_1 + alpha * n_2)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A1 - r_B) * (1 - alpha) * n_1 + (r_A2 - r_B) * alpha * n_2) + r_B)))

@njit(cache=True)
def f_2(n_2, n_1, r_A1, r_A2, r_B, d_A, d_B, beta):
    return (((d_B * (1 - n_2) * ((1 - beta) * r_A2 * n_2 + beta * r_A1 * n_1)) -
             (r_B * d_A * n_2 * (1 - ((1 - beta) * n_2 + beta * n_1)))) /
            (((d_A - d_B) * (n_1 + n_2) + 2 * d_B) *
             (((r_A2 - r_B) * (1 - beta) * n_2 + (r_A1 - r_B) * beta * n_1) + r_B)))

@njit(cache=True)
def integrate_euler(n01, n02, alpha_arr, beta_arr, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps):
    n_1 = np.zeros(num_steps)
    n_2 = np.zeros(num_steps)
    n_1[0] = n01
    n_2[0] = n02
    for i in range(1, num_steps):
        n_1[i] = n_1[i-1] + f_1(n_1[i-1], n_2[i-1], r_A1, r_A2, r_B, d_A, d_B, alpha_arr[i-1]) * dt
        n_2[i] = n_2[i-1] + f_2(n_2[i-1], n_1[i-1], r_A1, r_A2, r_B, d_A, d_B, beta_arr[i-1]) * dt
    return n_1, n_2


T_values = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000, 1500, 2000, 3000, 4000, 5000, 10000, 15000, 20000]
n01, n02 = 0.01, 0.00
t0, tf = 0.0, 100000.0
r_B, d_A, d_B = 1.0, 1.0, 1.0
dt = 0.01
alpha_0 = 0.1
beta0_values = [0.15, 0.2, 0.3]
num_steps = int((tf - t0) / dt)

r_A1_values = [1.6, 1.7, 1.8]  
r_A2_func = lambda r: 2.5 - r  

has_constant_migration = False

markers = {0.15: 'o', 0.2: 'd', 0.3: 'x'}
colors = {0.15: '#FF0000', 0.2: '#00AA00', 0.3: '#0000FF'}
inf_x_pos = {0.15: 1.3, 0.2: 1.7, 0.3: 2.2}


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0.20, hspace=0.28)

t = np.linspace(t0, tf, num_steps)
legend_handles, legend_labels = [], []

for idx, r_A1 in enumerate(r_A1_values):
    r_A2 = r_A2_func(r_A1)
    ax = axes[idx]
    ax.set_facecolor('white')
    all_y = []

    for beta0 in beta0_values:
        limiting = None

        avgs = []
        for T in T_values:
            alpha = alpha_0 * (np.cos(np.pi * t / T))**2
            beta = beta0 * (np.sin(np.pi * t / T))**2

            n1, n2 = integrate_euler(n01, n02, alpha, beta, r_A1, r_A2, r_B, d_A, d_B, dt, num_steps)
            y = (n1 + n2) / 2
            
            t_end = t[-1]
            mask = (t >= max(t_end - T, 60000.0)) & (t <= t_end)
            if mask.sum() > 1:
                Avg = np.trapz(y[mask], t[mask]) / T
            else:
                Avg = np.nan
            avgs.append(Avg if np.isfinite(Avg) else np.nan)

        valid = ~np.isnan(avgs)
        vT, vA = np.array(T_values)[valid], np.array(avgs)[valid]
        if len(vA) > 0:
            all_y.extend(vA)
            ax.plot(vT, vA, color=colors[beta0], lw=2.5, alpha=0.9)
            ax.scatter(vT, vA, marker=markers[beta0], color=colors[beta0],
                       s=85, edgecolors='black', linewidth=1.6)
            
            if idx == 0:
                h = Line2D([0], [0], color=colors[beta0], marker=markers[beta0],
                           lw=2.5, markersize=12, markerfacecolor=colors[beta0],
                           markeredgecolor='black', markeredgewidth=1.6)
                legend_handles.append(h)
                legend_labels.append(rf'\( \beta_0 = {beta0} \)')

    ys = [y for y in all_y if np.isfinite(y)]
    ax.set_ylim(min(ys)-0.08 if ys else 0, max(ys)+0.08 if ys else 1)
    ax.set_xscale('log')
    ax.set_xlim(min(T_values)*0.8, max(T_values)*1.5)

    delta_r = round(r_A1 - r_A2, 1)
    txt = rf'\( \delta r={delta_r} \)' + '\n' + rf'\( r_{{A1}}={r_A1} \)' + '\n' + rf'\( r_{{A2}}={r_A2:.1f} \)'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, fontsize=15,
            va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.9, edgecolor='gray', linewidth=0.8))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(3.5)
    ax.tick_params(axis='both', which='both', length=5, width=1.4, colors='black', labelsize=15)
    ax.grid(True, alpha=0.25, linestyle='--')

    if idx == 0:
        ax.set_ylabel(r'\( \bar{\phi} \)', fontsize=30)

fig.text(0.5, 0.02, r'\( T \)', fontsize=25, ha='center', fontfamily='serif', style='italic')

axes[2].legend(legend_handles, legend_labels, loc='lower right', framealpha=0.95,
               edgecolor='black', fancybox=False)

plt.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])
plt.savefig('Asymmetric_Out_of_Phase_beta_3.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()